In [8]:
import pandas as pd
import re

def clean_gutenberg_catalog(input_file_path, output_file_path, max_records=500):
    # 1. Load dataset using the input_file_path parameter
    df = pd.read_csv(input_file_path)
    
    # Strip extra whitespace from column names
    df.columns = df.columns.str.strip()

    # Rename 'Text#' header to 'ISBN' (keeping original values intact)
    if 'Text#' in df.columns:
        df.rename(columns={'Text#': 'ISBN'}, inplace=True)

    # 2. Filter for actual text/books (exclude audio/music/image entries)
    if 'Type' in df.columns:
        df = df[df['Type'].str.strip() == 'Text'].copy()

    # Slice the dataframe to the first max_records (500)
    df = df.head(max_records).copy()

    # 3. Clean 'Title'
    df['Title'] = df['Title'].astype(str).str.strip()

    # 4. Clean and Extract Author Metadata
    def parse_author(author_str):
        if pd.isna(author_str) or not str(author_str).strip():
            return pd.Series([None, None, None])
        
        # Extract lifespan years (e.g., 1809-1865)
        lifespan_match = re.search(r'(\d{4})?\s*[-–—]\s*(\d{4})?', str(author_str))
        birth_year = lifespan_match.group(1) if lifespan_match and lifespan_match.group(1) else None
        death_year = lifespan_match.group(2) if lifespan_match and lifespan_match.group(2) else None

        # Clean author name (remove dates and role markers like [Editor])
        clean_name = re.sub(r',?\s*\d{4}\s*[-–—]\s*\d{4}?', '', str(author_str))
        clean_name = re.sub(r'\[.*?\]', '', clean_name).strip()

        # Convert "Last, First" to "First Last"
        if ',' in clean_name:
            parts = [p.strip() for p in clean_name.split(',', 1)]
            clean_name = f"{parts[1]} {parts[0]}" if len(parts) == 2 else clean_name

        return pd.Series([clean_name, birth_year, death_year])

    if 'Authors' in df.columns:
        df[['Author_Name', 'Author_Birth_Year', 'Author_Death_Year']] = df['Authors'].apply(parse_author)
        df.drop(columns=['Authors'], inplace=True)

    # 5. Convert 'Issued' to string format (YYYY-MM-DD) for clean JSON serialization
    if 'Issued' in df.columns:
        df['Issued_Date'] = pd.to_datetime(df['Issued'], errors='coerce').dt.strftime('%Y-%m-%d')
        df.drop(columns=['Issued'], inplace=True)

    # 6. Clean Language
    if 'Language' in df.columns:
        df['Language'] = df['Language'].astype(str).str.strip()

    # 7. Reorder columns logically
    cols_order = [
        'ISBN', 'Title', 'Author_Name', 'Author_Birth_Year', 'Author_Death_Year', 
        'Language', 'Issued_Date', 'Type'
    ]
    
    existing_cols = [c for c in cols_order if c in df.columns]
    extra_cols = [c for c in df.columns if c not in existing_cols]
    df = df[existing_cols + extra_cols]

    # 8. Save dataset as JSON
    df.to_json(output_file_path, orient='records', indent=4)
    print(f"Successfully saved cleaned catalog to {output_file_path} ({len(df)} records)")

# Execute script
clean_gutenberg_catalog(
    input_file_path="pg_catalog.csv", 
    output_file_path="matts_cool_catalog.json", 
    max_records=500
)

Successfully saved cleaned catalog to matts_cool_catalog.json (500 records)
